In [1]:
import torch
from collections import OrderedDict

SRC = "/home/work/root/Meme_coin/ddc/work_dirs/pretrain_mitb5_f/best_mIoU_iter_75000.pth"
DST = "/home/work/root/Meme_coin/ddc/work_dirs/pretrain_mitb5_f/merge_fixed.pth"

ckpt = torch.load(SRC, map_location="cpu")
state = ckpt["state_dict"]

print(f"[SRC] total keys: {len(state)}")

new_state = OrderedDict()

for k, v in state.items():
    new_k = None

    # 1) 옛날 4→3 어댑터: backbone.adapter.*  →  backbone.rgbn_adapter.*
    if k.startswith("backbone.adapter."):
        tail = k[len("backbone.adapter."):]
        new_k = "backbone.rgbn_adapter." + tail

    # 2) 옛날 MiT-B5 본체: backbone.backbone.layers.*  →  backbone.mit.layers.*
    elif k.startswith("backbone.backbone.layers."):
        tail = k[len("backbone.backbone."):]  # 'layers.0.0.projection.weight'...
        new_k = "backbone.mit." + tail        # 'backbone.mit.layers.0.0.projection.weight'

    # 3) decode_head.* 는 버리고 싶으면 그냥 else: pass
    else:
        continue

    new_state[new_k] = v

print(f"[NEW] kept keys: {len(new_state)}")
print("  sample (first 10):")
for k in list(new_state.keys())[:10]:
    print("   ", k)

new_ckpt = {
    "state_dict": new_state,
    "meta": ckpt.get("meta", {})
}

torch.save(new_ckpt, DST)
print("[OK] Saved encoder-only checkpoint to:", DST)


[SRC] total keys: 981
[NEW] kept keys: 949
  sample (first 10):
    backbone.rgbn_adapter.weight
    backbone.mit.layers.0.0.projection.weight
    backbone.mit.layers.0.0.projection.bias
    backbone.mit.layers.0.0.norm.weight
    backbone.mit.layers.0.0.norm.bias
    backbone.mit.layers.0.1.0.norm1.weight
    backbone.mit.layers.0.1.0.norm1.bias
    backbone.mit.layers.0.1.0.attn.attn.in_proj_weight
    backbone.mit.layers.0.1.0.attn.attn.in_proj_bias
    backbone.mit.layers.0.1.0.attn.attn.out_proj.weight
[OK] Saved encoder-only checkpoint to: /home/work/root/Meme_coin/ddc/work_dirs/pretrain_mitb5_f/merge_fixed.pth
